QUESTÃO IV:<br>

O SSD se destaca na detecção de objetos com detalhes específicos e padrões bem definidos nas imagens. Quando analisei, percebi que ele é particularmente bom em situações onde precisamos reconhecer estruturas com características bem marcadas, como em imagens de satélite onde dá pra identificar estradas ou em cenários urbanos onde precisamos diferenciar objetos com formatos semelhantes. Ele usa várias camadas de detecção que permitem capturar características em diferentes escalas, o que ajuda bastante na precisão.

O YOLO, por outro lado, é ideal para quando a gente precisa de velocidade em aplicações de tempo real. Ele processa a imagem toda de uma vez só, como o próprio nome já diz "You Only Look Once". Quando testei, notei que ele é bem mais rápido que outras opções, o que é ótimo pra trabalhar com vídeos ou câmeras ao vivo. A diferença principal tá na arquitetura mesmo - o YOLO divide a imagem em regiões e faz todas as previsões simultaneamente, enquanto o SSD vai analisando em múltiplas camadas. O YOLO acaba sacrificando um pouco da precisão pra ganhar essa velocidade toda, especialmente com objetos menores, onde o SSD costuma se sair melhor

Os vídeos de saídas dos modelos estão no drive institucional, caso queira executar ov vídeo original está junto com os ouputs, só extrair na root dir do algoritmo: [\[link do drive para vídeos\]](https://drive.google.com/file/d/1lKVLym9BSnTFmdfBP17DFHSHSxKcnh7s/view?usp=sharing)

In [131]:
import torch
import torchvision
from torchvision.models.detection import ssd300_vgg16
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
import requests
from tqdm import tqdm
from ultralytics import YOLO
import traceback
from torchvision import transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [132]:
os.makedirs('models', exist_ok=True)
def download_file(url, filename):
    if os.path.exists(filename):
        return
    
    print(f"Baixando {filename}...")
    response = requests.get(url, stream=True)
    total_size = int(response.headers.get('content-length', 0))
    block_size = 1024
    progress_bar = tqdm(total=total_size, unit='iB', unit_scale=True)
    
    with open(filename, 'wb') as file:
        for data in response.iter_content(block_size):
            progress_bar.update(len(data))
            file.write(data)
    
    progress_bar.close()

In [133]:
# Download modelo YOLOv4
yolo_cfg_url = "https://raw.githubusercontent.com/AlexeyAB/darknet/master/cfg/yolov4.cfg"
yolo_weights_url = "https://github.com/AlexeyAB/darknet/releases/download/darknet_yolo_v3_optimal/yolov4.weights"
yolo_coco_names_url = "https://raw.githubusercontent.com/AlexeyAB/darknet/master/data/coco.names"
# yolo_v8 = "https://github.com/ultralytics/assets/releases/download/v8.1.0/yolov8n.pt"

# download_file(yolo_v8, "models/yolov8.pt")
download_file(yolo_cfg_url, "models/yolov4.cfg")
download_file(yolo_weights_url, "models/yolov4.weights")
download_file(yolo_coco_names_url, "models/coco.names")



In [134]:
# Carregar modelo YOLO (COCO classes dataset)
net_yolo = cv2.dnn.readNetFromDarknet("models/yolov4.cfg", "models/yolov4.weights")
net_yolo.setPreferableBackend(cv2.dnn.DNN_BACKEND_OPENCV)
net_yolo.setPreferableTarget(cv2.dnn.DNN_TARGET_CPU)

with open("models/coco.names", 'r') as f:
    classes_yolo = [line.strip() for line in f.readlines()]

In [ ]:
def process_video_yolo(video_path, output_path, confidence_threshold=0.5, nms_threshold=0.4):
    # Veículos que queremos detectar (índices das classes COCO)
    vehicle_classes = ['car', 'truck', 'bus', 'motorbike']
    vehicle_indices = [classes_yolo.index(cls) for cls in vehicle_classes]
    cap = cv2.VideoCapture(video_path)
    
    # Verificar se o vídeo foi aberto corretamente
    if not cap.isOpened():
        return [], []

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # Usar codec compatível macOS
    fourcc = cv2.VideoWriter_fourcc(*'MJPG')
    output_path_avi = output_path if output_path.endswith('.avi') else output_path.replace('.mp4', '.avi')
    output_abs = os.path.abspath(output_path_avi)
    print(f"Salvando vídeo YOLO em: {output_abs}")
    
    writer = cv2.VideoWriter(output_abs, fourcc, fps, (width, height))
    if not writer.isOpened():
        return [], []
    
    frame_numbers = []
    vehicle_counts = []    
    frame_idx = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        # Criar blob de input da rede neural
        blob = cv2.dnn.blobFromImage(frame, 1/255.0, (416, 416), swapRB=True, crop=False)
        net_yolo.setInput(blob)
        
        # Obter layers saída
        layer_names = net_yolo.getLayerNames()
        output_layers = [layer_names[i - 1] for i in net_yolo.getUnconnectedOutLayers().flatten()]
        outputs = net_yolo.forward(output_layers)
        
        boxes = []
        confidences = []
        class_ids = []
        
        for output in outputs:
            for detection in output:
                scores = detection[5:]
                class_id = np.argmax(scores)
                confidence = scores[class_id]
                
                if class_id in vehicle_indices and confidence > confidence_threshold:
                    center_x = int(detection[0] * width)
                    center_y = int(detection[1] * height)
                    box_width = int(detection[2] * width)
                    box_height = int(detection[3] * height)

                    x = int(center_x - box_width / 2)
                    y = int(center_y - box_height / 2)
                    
                    boxes.append([x, y, box_width, box_height])
                    confidences.append(float(confidence))
                    class_ids.append(class_id)
        
        indices = cv2.dnn.NMSBoxes(boxes, confidences, confidence_threshold, nms_threshold)
        vehicle_count = len(indices)

        frame_numbers.append(frame_idx)
        vehicle_counts.append(vehicle_count)
        
        for i in indices.flatten():
            x, y, w, h = boxes[i]
            class_id = class_ids[i]
            label = f"{classes_yolo[class_id]}: {confidences[i]:.2f}"
            cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
            cv2.putText(frame, label, (x, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
        
        cv2.putText(frame, f"Veiculos: {vehicle_count}", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
        writer.write(frame)
        
        frame_idx += 1
        if frame_idx % 10 == 0:
            print(f"YOLO: Processado {frame_idx}/{frame_count} frames")
    
    cap.release()
    writer.release()
    return frame_numbers, vehicle_counts

In [ ]:
def load_pytorch_ssd():
    # Carregar modelo SSD300 com backbone VGG16 pré-treinado com COCO
    model = ssd300_vgg16(pretrained=True)
    model.eval()
    model.to(device)

    coco_names = [
        'car', 'motorcycle', 'airplane', 'bus',
        'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'N/A', 'stop sign',
        'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow',
        'elephant', 'bear', 'zebra', 'giraffe', 'N/A', 'backpack', 'umbrella', 'N/A', 'N/A',
        'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball',
        'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket',
        'bottle', 'N/A', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl',
        'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza',
        'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'N/A', 'dining table',
        'N/A', 'N/A', 'toilet', 'N/A', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone',
        'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'N/A', 'book',
        'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush'
    ]
    
    return model, coco_names

def process_video_pytorch_ssd(video_path, output_path, confidence_threshold=0.5):
    # Load model do pytorch ssd
    model, coco_names = load_pytorch_ssd()
    
    # Veículos que queremos detectar
    vehicle_classes = ['car', 'bus', 'truck', 'motorcycle']
    vehicle_indices = [coco_names.index(cls) for cls in vehicle_classes]

    transform = transforms.Compose([
        transforms.ToTensor(),
    ])

    cap = cv2.VideoCapture(video_path)
    
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    fourcc = cv2.VideoWriter_fourcc('M', 'J', 'P', 'G')
    output_abs = os.path.abspath(output_path)
    print(f"Salvando vídeo SSD em: {output_abs}")
    
    writer = cv2.VideoWriter(output_abs, fourcc, fps, (width, height))
    if not writer.isOpened():
        return [], []
    
    frame_numbers = []
    vehicle_counts = []
    frame_idx = 0
    
    print("Iniciando PyTorch SSD...")
    
    # Processar frames
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        try:
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            img_tensor = transform(rgb_frame).to(device)
            img_tensor = img_tensor.unsqueeze(0)
            
            # Inferência
            with torch.no_grad():
                detections = model(img_tensor)
            
            boxes = []
            confidences = []
            class_ids = []
            
            for i in range(len(detections[0]['boxes'])):
                confidence = detections[0]['scores'][i].item()
                class_id = detections[0]['labels'][i].item()
                
                if confidence > confidence_threshold and class_id in vehicle_indices:
                    box = detections[0]['boxes'][i].cpu().numpy().astype(int)
                    
                    # Calcular coordenadas (x, y, w, h)
                    x, y, x2, y2 = box
                    w = x2 - x
                    h = y2 - y
                    
                    boxes.append([x, y, w, h])
                    confidences.append(float(confidence))
                    class_ids.append(class_id)
            
            vehicle_count = len(boxes)
            
            frame_numbers.append(frame_idx)
            vehicle_counts.append(vehicle_count)

            for i in range(len(boxes)):
                x, y, w, h = boxes[i]
                class_id = class_ids[i]
                class_name = coco_names[class_id]
                label = f"{class_name}: {confidences[i]:.2f}"
                
                cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 0, 255), 2)
                cv2.putText(frame, label, (x, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)
            
            cv2.putText(frame, f"Veiculos: {vehicle_count}", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
            writer.write(frame)
            
            if frame_idx % 10 == 0:
                print(f"PyTorch SSD: Processado {frame_idx}/{frame_count} frames")
            
        except Exception as e:
            print(f"Erro ao processar frame {frame_idx}: {e}")
            traceback_info = traceback.format_exc()
            print(traceback_info)
        
        frame_idx += 1
    
    cap.release()
    writer.release()
    print("Processamento PyTorch SSD concluído")
    return frame_numbers, vehicle_counts

In [ ]:
video_path = "bridge.mp4"
output_yolo_path = "output_yolo.avi"
output_ssd_path = "output_ssd.avi"

print("Iniciando YOLO/SSD")
yolo_frames, yolo_counts = process_video_yolo(video_path, output_yolo_path)
torch_ssd_frames, torch_ssd_counts = process_video_pytorch_ssd(video_path, output_ssd_path)

print("Finalizado")

Iniciando YOLO/SSD
Salvando vídeo YOLO em: /Users/luryand/Documents/VC/codes/t4/output_yolo.avi


[ WARN:0@28580.139] global cap.cpp:781 open VIDEOIO(CV_IMAGES): raised OpenCV exception:

OpenCV(4.11.0) /private/var/folders/zb/fsvx994s2g54rx2zz2nh6sp40000gn/T/pip-install-1kisax0g/opencv-python_73e27ae3737a46c3853d2d66fa1bf6ce/opencv/modules/videoio/src/cap_images.cpp:415: error: (-215:Assertion failed) !filename_pattern.empty() in function 'CvVideoWriter_Images'




YOLO: Processado 10/577 frames
YOLO: Processado 20/577 frames
YOLO: Processado 30/577 frames
YOLO: Processado 40/577 frames
YOLO: Processado 50/577 frames
YOLO: Processado 60/577 frames
YOLO: Processado 70/577 frames
YOLO: Processado 80/577 frames
YOLO: Processado 90/577 frames
YOLO: Processado 100/577 frames
YOLO: Processado 110/577 frames
YOLO: Processado 120/577 frames
YOLO: Processado 130/577 frames
YOLO: Processado 140/577 frames
YOLO: Processado 150/577 frames
YOLO: Processado 160/577 frames
YOLO: Processado 170/577 frames
YOLO: Processado 180/577 frames
YOLO: Processado 190/577 frames
YOLO: Processado 200/577 frames
YOLO: Processado 210/577 frames
YOLO: Processado 220/577 frames
YOLO: Processado 230/577 frames
YOLO: Processado 240/577 frames
YOLO: Processado 250/577 frames
YOLO: Processado 260/577 frames
YOLO: Processado 270/577 frames
YOLO: Processado 280/577 frames
YOLO: Processado 290/577 frames
YOLO: Processado 300/577 frames
YOLO: Processado 310/577 frames
YOLO: Processado 